# AIC 2026 — Chuẩn bị dữ liệu trên Google Drive

Mở notebook này bằng **Google Colab**, chọn **Runtime → Run all**, rồi đăng nhập Google Drive khi được hỏi.

Notebook sẽ:

1. Kết nối Google Drive.
2. Tạo `MyDrive/AIC2026/{raw,indices,candidates,submission}`.
3. Tải ba ZIP tối thiểu từ máy chủ BTC thẳng vào Drive.
4. Tiếp tục một lượt tải bị gián đoạn thay vì tải lại từ đầu.
5. Kiểm tra đúng dung lượng từng ZIP.
6. Copy và giải nén ba ZIP vào `/content/aic_work` để xử lý nhanh trong phiên Colab hiện tại.

> ZIP được giữ nguyên trên Drive. Dữ liệu giải nén trong `/content` là tạm thời và sẽ mất khi Colab kết thúc phiên.


In [1]:
# 1) Kết nối Google Drive và tạo cấu trúc thư mục
from pathlib import Path

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "Hãy mở file này bằng Google Colab (https://colab.research.google.com) rồi chọn Runtime > Run all."
    ) from exc

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/AIC2026")
RAW_DIR = DRIVE_ROOT / "raw"
INDICES_DIR = DRIVE_ROOT / "indices"
CANDIDATES_DIR = DRIVE_ROOT / "candidates"
SUBMISSION_DIR = DRIVE_ROOT / "submission"
WORK_DIR = Path("/content/aic_work")

for directory in (RAW_DIR, INDICES_DIR, CANDIDATES_DIR, SUBMISSION_DIR, WORK_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Drive root : {DRIVE_ROOT}")
print(f"Runtime dir: {WORK_DIR}")


Mounted at /content/drive
Drive root : /content/drive/MyDrive/AIC2026
Runtime dir: /content/aic_work


In [2]:
# 2) Tải ba ZIP tối thiểu. Cell có thể chạy lại an toàn.
import subprocess

CORE_FILES = {
    "clip-features-32-aic25-b1.zip": {
        "url": "https://aic-data.ledo.io.vn/clip-features-32-aic25-b1.zip",
        "expected_bytes": 168_173_583,
    },
    "map-keyframes-aic25-b1.zip": {
        "url": "https://aic-data.ledo.io.vn/map-keyframes-aic25-b1.zip",
        "expected_bytes": 1_604_272,
    },
    "media-info-aic25-b1.zip": {
        "url": "https://aic-data.ledo.io.vn/media-info-aic25-b1.zip",
        "expected_bytes": 1_111_755,
    },
}

def download_to_drive(filename, spec):
    destination = RAW_DIR / filename
    expected = spec["expected_bytes"]

    if destination.exists() and destination.stat().st_size == expected:
        print(f"✓ Đã có: {filename} ({expected / 1e6:.2f} MB)")
        return destination

    if destination.exists():
        print(
            f"↻ Tiếp tục: {filename} "
            f"({destination.stat().st_size / 1e6:.2f}/{expected / 1e6:.2f} MB)"
        )
    else:
        print(f"↓ Đang tải: {filename} ({expected / 1e6:.2f} MB)")

    subprocess.run(
        ["wget", "-c", "--show-progress", spec["url"]],
        cwd=str(RAW_DIR),
        check=True,
    )

    actual = destination.stat().st_size
    if actual != expected:
        raise RuntimeError(
            f"Dung lượng {filename} không đúng: {actual:,} bytes; "
            f"mong đợi {expected:,} bytes. Hãy chạy lại cell để tiếp tục tải."
        )

    print(f"✓ Hoàn tất: {filename} ({actual / 1e6:.2f} MB)")
    return destination

downloaded_files = [
    download_to_drive(filename, spec)
    for filename, spec in CORE_FILES.items()
]


↓ Đang tải: clip-features-32-aic25-b1.zip (168.17 MB)
✓ Hoàn tất: clip-features-32-aic25-b1.zip (168.17 MB)
↓ Đang tải: map-keyframes-aic25-b1.zip (1.60 MB)
✓ Hoàn tất: map-keyframes-aic25-b1.zip (1.60 MB)
↓ Đang tải: media-info-aic25-b1.zip (1.11 MB)
✓ Hoàn tất: media-info-aic25-b1.zip (1.11 MB)


In [3]:
# 3) Kiểm tra ZIP và giải nén vào ổ tạm nhanh của Colab
import shutil
import zipfile

def safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    destination_root = destination.resolve()

    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = (destination / member.filename).resolve()
            if member_path != destination_root and destination_root not in member_path.parents:
                raise RuntimeError(f"ZIP chứa đường dẫn không an toàn: {member.filename}")
        archive.extractall(destination)
        return len(archive.infolist())

extracted_dirs = {}
for drive_zip in downloaded_files:
    if not zipfile.is_zipfile(drive_zip):
        raise RuntimeError(f"Không phải ZIP hợp lệ: {drive_zip}")

    local_zip = WORK_DIR / drive_zip.name
    print(f"→ Copy {drive_zip.name} xuống runtime...")
    shutil.copy2(drive_zip, local_zip)

    destination = WORK_DIR / drive_zip.stem
    member_count = safe_extract(local_zip, destination)
    extracted_dirs[drive_zip.name] = destination
    print(f"✓ Đã giải nén {member_count:,} mục vào {destination}")


→ Copy clip-features-32-aic25-b1.zip xuống runtime...
✓ Đã giải nén 873 mục vào /content/aic_work/clip-features-32-aic25-b1
→ Copy map-keyframes-aic25-b1.zip xuống runtime...
✓ Đã giải nén 874 mục vào /content/aic_work/map-keyframes-aic25-b1
→ Copy media-info-aic25-b1.zip xuống runtime...
✓ Đã giải nén 873 mục vào /content/aic_work/media-info-aic25-b1


In [4]:
# 4) Báo cáo cuối cùng
print("\n=== AIC 2026 DATA SETUP COMPLETE ===")
print(f"ZIP lưu lâu dài tại: {RAW_DIR}")
print(f"Dữ liệu làm việc tạm tại: {WORK_DIR}")
print("\nCác thư mục đã giải nén:")
for filename, directory in extracted_dirs.items():
    print(f"- {filename} -> {directory}")
print("\nCó thể đóng notebook sau khi thấy dòng DATA SETUP COMPLETE.")



=== AIC 2026 DATA SETUP COMPLETE ===
ZIP lưu lâu dài tại: /content/drive/MyDrive/AIC2026/raw
Dữ liệu làm việc tạm tại: /content/aic_work

Các thư mục đã giải nén:
- clip-features-32-aic25-b1.zip -> /content/aic_work/clip-features-32-aic25-b1
- map-keyframes-aic25-b1.zip -> /content/aic_work/map-keyframes-aic25-b1
- media-info-aic25-b1.zip -> /content/aic_work/media-info-aic25-b1

Có thể đóng notebook sau khi thấy dòng DATA SETUP COMPLETE.


## Sau khi chạy xong

- Ba ZIP vẫn nằm trong `MyDrive/AIC2026/raw`.
- Các thư mục giải nén trong `/content/aic_work` chỉ tồn tại trong phiên Colab hiện tại.
- Chưa tải Objects, Keyframes hoặc Videos. Chỉ tải các gói này sau khi CLIP đã tìm được nhóm Lxx ứng viên.
- Nếu quá trình tải bị ngắt, kết nối lại Drive và chọn **Run all**; `wget -c` sẽ tiếp tục từ phần đã có.


In [5]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

WORK_DIR = Path("/content/aic_work")

npy_files = sorted(WORK_DIR.rglob("*.npy"))
csv_files = sorted(WORK_DIR.rglob("*.csv"))
json_files = sorted(WORK_DIR.rglob("*.json"))

print("Số file .npy :", len(npy_files))
print("Số file .csv :", len(csv_files))
print("Số file .json:", len(json_files))

print("\n5 NPY đầu:")
for path in npy_files[:5]:
    print("-", path.relative_to(WORK_DIR))

print("\n5 CSV đầu:")
for path in csv_files[:5]:
    print("-", path.relative_to(WORK_DIR))

print("\n5 JSON đầu:")
for path in json_files[:5]:
    print("-", path.relative_to(WORK_DIR))

if npy_files:
    sample_embedding = np.load(npy_files[0], mmap_mode="r")
    print("\n=== SAMPLE EMBEDDING ===")
    print("File :", npy_files[0])
    print("Shape:", sample_embedding.shape)
    print("Dtype:", sample_embedding.dtype)

if csv_files:
    sample_map = pd.read_csv(csv_files[0])
    print("\n=== SAMPLE KEYFRAME MAP ===")
    print("File   :", csv_files[0])
    print("Columns:", sample_map.columns.tolist())
    display(sample_map.head())

if json_files:
    with open(json_files[0], encoding="utf-8-sig") as file:
        sample_media = json.load(file)

    print("\n=== SAMPLE MEDIA INFO ===")
    print("File:", json_files[0])
    print("Type:", type(sample_media).__name__)
    if isinstance(sample_media, dict):
        print("Keys:", list(sample_media.keys()))
        print(json.dumps(sample_media, ensure_ascii=False, indent=2)[:2000])

Số file .npy : 873
Số file .csv : 873
Số file .json: 873

5 NPY đầu:
- clip-features-32-aic25-b1/clip-features-32/L21_V001.npy
- clip-features-32-aic25-b1/clip-features-32/L21_V002.npy
- clip-features-32-aic25-b1/clip-features-32/L21_V003.npy
- clip-features-32-aic25-b1/clip-features-32/L21_V005.npy
- clip-features-32-aic25-b1/clip-features-32/L21_V006.npy

5 CSV đầu:
- map-keyframes-aic25-b1/map-keyframes/L21_V001.csv
- map-keyframes-aic25-b1/map-keyframes/L21_V002.csv
- map-keyframes-aic25-b1/map-keyframes/L21_V003.csv
- map-keyframes-aic25-b1/map-keyframes/L21_V005.csv
- map-keyframes-aic25-b1/map-keyframes/L21_V006.csv

5 JSON đầu:
- media-info-aic25-b1/media-info/L21_V001.json
- media-info-aic25-b1/media-info/L21_V002.json
- media-info-aic25-b1/media-info/L21_V003.json
- media-info-aic25-b1/media-info/L21_V005.json
- media-info-aic25-b1/media-info/L21_V006.json

=== SAMPLE EMBEDDING ===
File : /content/aic_work/clip-features-32-aic25-b1/clip-features-32/L21_V001.npy
Shape: (307, 5

,n,pts_time,fps,frame_idx
0,1,0.0000,30.0,0
1,2,3.0000,30.0,90
2,3,8.7000,30.0,261
3,4,11.7333,30.0,351
4,5,13.7000,30.0,411



=== SAMPLE MEDIA INFO ===
File: /content/aic_work/media-info-aic25-b1/media-info/L21_V001.json
Type: dict
Keys: ['author', 'channel_id', 'channel_url', 'description', 'keywords', 'length', 'publish_date', 'thumbnail_url', 'title', 'watch_url']
{
  "author": "60 Giây Official",
  "channel_id": "UCRjzfa1E0gA50lvDQipbDMg",
  "channel_url": "https://www.youtube.com/channel/UCRjzfa1E0gA50lvDQipbDMg",
  "description": "60 Giây Sáng - Ngày 01/08/2024 - HTV Tin Tức Mới Nhất 2024\n► Đăng ký KÊNH để xem Tin Tức Mới Nhất: https://bit.ly/2HoUna4\nKênh Tin Tức Thời Sự 60 Giây Là Kênh Tổng Hợp Tin Tức - Sự Kiện - Giải trí Nhanh Nhất Việt Nam.\n\n©️ Bản quyền Tin Tức thuộc HTV - Đài Truyền hình TP.HCM\n©️ The copyright of this video belongs to HTV - Ho Chi Minh City Television\n#TintucthoisuVietnam\n#HTVTintuc #HTVnews #HTV60s #60Giay #60s\n------------\nXem TV online tại Truyền hình HTVC:\n✅ Web / Wap mobile :\n► https://hplus.com.vn\n► https://htvc.com.vn/\n► https://htvc.tv/\n✅ App mobile/TV HTVC